# Dataset


## Setup


### Imports


In [ ]:
import os
import subprocess
import xml.etree.ElementTree as ET

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

### Environment


Here we set up the environment for the pipeline. We need to define where the SUMO library is located and add the `bin` and `tools` directories to the system path. This is necessary to be able to use the SUMO tools and libraries in the pipeline.


In [ ]:
os.environ["SUMO_HOME"] = os.path.join(os.environ["CONDA_PREFIX"], "Lib", "site-packages", "sumo")

SUMO_HOME = os.environ["SUMO_HOME"]

os.environ["PATH"] += os.pathsep + os.path.join(SUMO_HOME, "bin")
os.environ["PATH"] += os.pathsep + os.path.join(SUMO_HOME, "tools")

### Constants


For the traffic generation periods, we tried to have as realistic as possible traffic patterns. This includes a morning rush hour, a lunch break, and an evening rush hour. A useful reference for traffic patterns in Athens is the [Athens Mobility Observatory](https://amob.ntua.gr/traffic/).


In [ ]:
np.random.seed(42)

DATASET_DIR = os.path.join("athens-10h")

OSM_WEB_WIZARD_PATH = os.path.join(SUMO_HOME, "tools", "osmWebWizard.py")
RANDOM_TRIPS_PATH = os.path.join(SUMO_HOME, "tools", "randomTrips.py")
DUAROUTER_PATH = os.path.join(SUMO_HOME, "bin", "duarouter.exe")
XML_2_CSV_PATH = os.path.join(SUMO_HOME, "tools", "xml", "xml2csv.py")

NETWORK_PATH = os.path.join(DATASET_DIR, "osm.net.xml.gz")
FCD_TRAIN_DATASET_PATH = os.path.join(DATASET_DIR, "train-fcd.xml")
FCD_TEST_DATASET_PATH = os.path.join(DATASET_DIR, "test-fcd.xml")
CLOSURE_TRAIN_FCD_DATASET_PATH = os.path.join(DATASET_DIR, "closure-train-fcd.xml")
CLOSURE_TEST_FCD_DATASET_PATH = os.path.join(DATASET_DIR, "closure-test-fcd.xml")
RAIN_TRAIN_FCD_DATASET_PATH = os.path.join(DATASET_DIR, "rain-train-fcd.xml")
RAIN_TEST_FCD_DATASET_PATH = os.path.join(DATASET_DIR, "rain-test-fcd.xml")

TRAIN_TRAFFIC_GENERATION_PERIODS = [0.45, 0.50, 0.65, 0.75, 0.80, 0.80, 0.75, 0.55, 0.50, 0.55]
TEST_TRAFFIC_GENERATION_PERIODS = [p * np.random.uniform(0.98, 1.02) for p in TRAIN_TRAFFIC_GENERATION_PERIODS]

## Network Generation


To generate the network, we can use the [osmWebWizard](https://sumo.dlr.de/docs/Tutorials/OSMWebWizard.html). For options, we selected the following:

- Position: Athens (for more details, see `dataset/athens-10h/images/athens-map.png`)
- Duration: 36000 (in seconds, effectively 10 hours)
- Add Polygons: Enabled
- Car-only Network: Enabled
- Random Traffic Generation: Disabled All


In [ ]:
!python $SUMO_HOME/tools/osmWebWizard.py

## Fixed Routes


Following, we have defined a few fixed routes. These routes are used on the test dataset, so that we have specific ground truths for visualization and evaluation purposes. These routes are fixed and are not rerouted during the simulation.

The flows, contained in the `dataset/athens-10h/fixed.flows.xml` file, were firstly defined using the [netedit](https://sumo.dlr.de/docs/Netedit/index.html) tool of SUMO. For now we only have one trip, with two different routes. The trip is **Omonoia to Evangelismos** and the routes are **via Akadimias** (Route A) and **via Stadiou** (Route B). The routes are as follows:

- **Route A - Via Akadimias**: 23182962 (Stadiou) > 260124786#0 (Akadimias) > 1209362820 (Pl. Filikis Eterias) > 299645496 (Marasli) > 169130585 (Leof. Vasilissis Sofias)
- **Route B - Via Stadiou**: 23182962 (Stadiou) > 299506410#0 (Stadiou) > 221139568 (Leof. Vasilissis Amalias) > -820421378#1 (Leof. Vasilissis Sofias) > 169130585 (Leof. Vasilissis Sofias)

After defining the routes, we can use the `duarouter` tool to generate the fixed routes that will be used in the simulation.


In [ ]:
!netedit athens-10h/osm.net.xml.gz

In [ ]:
def generate_fixed_routes(prefix):
    fixed_flows_file = os.path.join(DATASET_DIR, f"{prefix}.flows.xml")
    fixed_routes_file = os.path.join(DATASET_DIR, f"{prefix}.rou.xml")
    fixed_routes_alt_file = os.path.join(DATASET_DIR, f"{prefix}.rou.alt.xml")

    command = [
        DUAROUTER_PATH,
        "--net-file",
        NETWORK_PATH,
        "--route-files",
        fixed_flows_file,
        "--output-file",
        fixed_routes_file,
    ]

    print("Executing:", " ".join(command))
    result = subprocess.run(command, capture_output=True, text=True)

    if result.stderr:
        print("Errors/Warnings from duarouter:")
        print(result.stderr)

    if os.path.exists(fixed_routes_file) and os.path.exists(fixed_routes_alt_file):
        os.remove(fixed_routes_alt_file)
        print("Success:", fixed_routes_file)
    else:
        print("Failed:", fixed_routes_file)

In [ ]:
print("Generating fixed routes...")
generate_fixed_routes("fixed")

## Preparation


### Random Trips


To generate random trips, we can use the [randomTrips.py](https://sumo.dlr.de/docs/Tools/Trip.html) script from the SUMO tools.

This script generates random trips based on the network and the specified parameters. The generated trips are saved in a file that can be used for simulation.

Here, we can specify the network file to use, the beginning and end of the simulation, the traffic generation periods to use, the output file name, a random seed for reproducibility and a flag to validate the trips using `duarouter`.

The traffic generation periods are passed as a comma-separated string. The script will then split the duration of the simulation into equal subintervals and generate trips accordingly.


In [ ]:
def generate_random_trips(prefix, traffic_generation_periods, seed=42, simulation_start=0, simulation_end=36000):
    traffic_generation_periods_str = ",".join(str(v) for v in traffic_generation_periods)
    trips_file = os.path.join(DATASET_DIR, f"{prefix}.trips.xml")
    routes_temp_file = "routes.rou.xml"

    command = [
        "python",
        RANDOM_TRIPS_PATH,
        "--net-file",
        NETWORK_PATH,
        "--begin",
        str(simulation_start),
        "--end",
        str(simulation_end),
        "--period",
        traffic_generation_periods_str,
        "-o",
        trips_file,
        "--seed",
        str(seed),
        "--validate",
    ]

    print("Executing:", " ".join(command))
    result = subprocess.run(command, capture_output=True, text=True)
    if result.stderr:
        print("Errors/Warnings from randomTrips:")
        print(result.stderr)

    if os.path.exists(trips_file) and os.path.exists(routes_temp_file):
        os.remove(routes_temp_file)
        print("Success:", trips_file)
    else:
        print("Failed:", trips_file)

In [ ]:
print("Generating train random trips...")
generate_random_trips("train", TRAIN_TRAFFIC_GENERATION_PERIODS, 42)
print("Generating test random trips...")
generate_random_trips("test", TEST_TRAFFIC_GENERATION_PERIODS, 123)

### Trip IDs


Some trips are not valid, meaning that they are not able to be routed when using `duarouter`. Here we adjust the trip IDs to be continuous, essentially removing the gaps created by the missing invalid trips.


In [ ]:
def update_trip_ids(prefix):
    trips_file = os.path.join(DATASET_DIR, f"{prefix}.trips.xml")
    if not os.path.exists(trips_file):
        print(f"Trips file not found: {trips_file}")
        return

    tree = ET.parse(trips_file)
    root = tree.getroot()

    trip_id = 0
    for trip in root.findall("trip"):
        trip.set("id", str(trip_id))
        trip_id += 1

    tree.write(trips_file)
    print("Updated:", trips_file)

In [ ]:
print("Updating train trip IDs...")
update_trip_ids("train")
print("Updating test trip IDs...")
update_trip_ids("test")

### Vehicle Types


For the vehicle types, we ended up going with just cars. We have also defined a few more vehicle types, but they are not used in the dataset. These can be found on the `dataset/athens-10h/vtypes.xml` file.

The vehicle types and their parameters, contained in the file above, are based on the following resources:

- [SUMO Vehicle Types](https://sumo.dlr.de/docs/Definition_of_Vehicles,_Vehicle_Types,_and_Routes.html#vehicle_types)
- [SUMO Vehicle Type Parameter Defaults](https://sumo.dlr.de/docs/Vehicle_Type_Parameter_Defaults.html)
- [SUMO HBEFA3-based Emissions](https://sumo.dlr.de/docs/Models/Emissions/HBEFA3-based.html)

Here, we update the vehicle types for the random trips and if we have a test dataset, we also update the vehicle types for the fixed routes, since they will be used in the simulation.


In [ ]:
def update_vehicle_types(prefix, vehicle_type="car"):
    trips_file = os.path.join(DATASET_DIR, f"{prefix}.trips.xml")
    if not os.path.exists(trips_file):
        print(f"Trips file not found: {trips_file}")
        return

    tree = ET.parse(trips_file)
    root = tree.getroot()

    for trip in root.findall("trip"):
        trip.set("type", vehicle_type)

    tree.write(trips_file)
    print("Updated:", trips_file)

    if "test" not in prefix:
        return

    fixed_routes_file = os.path.join(DATASET_DIR, "fixed.rou.xml")
    if not os.path.exists(fixed_routes_file):
        print(f"Fixed routes file not found: {fixed_routes_file}")
        return

    tree = ET.parse(fixed_routes_file)
    root = tree.getroot()

    for vehicle in root.findall("vehicle"):
        vehicle.set("type", vehicle_type)

    tree.write(fixed_routes_file)
    print("Updated:", fixed_routes_file)

In [ ]:
print("Updating train vehicle types...")
update_vehicle_types("train")
print("Updating test vehicle types...")
update_vehicle_types("test")

## Simulation


For the simulation, we can use either the command line [sumo](https://sumo.dlr.de/docs/sumo.html) or launch the [sumo-gui](https://sumo.dlr.de/docs/sumo-gui.html) to have a visual representation of the simulation.


In [ ]:
!sumo -c athens-10h/train.sumocfg

In [ ]:
!sumo -c athens-10h/test.sumocfg

## Exploratory Data Analysis


### XML Parsing


Here we parse the XML FCD output generated by the SUMO simulation. The files is converted to csv format for further usage later, but for now we use the XML format for speed and simplicity.


In [ ]:
def parse_fcd_output_xml(file_path):
    records = []
    context = ET.iterparse(file_path, events=("end",))
    for _, elem in context:
        if elem.tag == "timestep":
            time = float(elem.get("time"))
            for veh in elem.findall("vehicle"):
                speed_ms = float(veh.get("speed"))
                records.append(
                    {
                        "timestep_time": time,
                        "vehicle_id": veh.get("id"),
                        "x": float(veh.get("x")),
                        "y": float(veh.get("y")),
                        "vehicle_type": veh.get("type"),
                        "speed_kmh": speed_ms * 3.6,
                        "acceleration": float(veh.get("acceleration")),
                        "odometer": float(veh.get("odometer")),
                    }
                )
            elem.clear()
    return pd.DataFrame(records)

In [ ]:
print("Parsing train FCD output...")
train_df = parse_fcd_output_xml(FCD_TRAIN_DATASET_PATH)
print("Parsing test FCD output...")
test_df = parse_fcd_output_xml(FCD_TEST_DATASET_PATH)

### Overview


In [ ]:
train_df = train_df.dropna().reset_index(drop=True)
test_df = test_df.dropna().reset_index(drop=True)

train_df = train_df[train_df["timestep_time"] < 36000].reset_index(drop=True)
test_df = test_df[test_df["timestep_time"] < 36000].reset_index(drop=True)

for df, df_name in [(train_df, "train"), (test_df, "test")]:
    print(f"Dataset: {df_name}")
    print(f"Data shape: {df.shape}")
    print(f"Average speed: {df['speed_kmh'].mean():.2f} km/h")
    print(f"Average distance (max odometer): {df.groupby('vehicle_id')['odometer'].max().mean():.2f} m")
    print(f"Unique vehicles: {df['vehicle_id'].nunique()}")
    print(f"Time span: {df['timestep_time'].min():.2f} s to {df['timestep_time'].max():.2f} s")
    if df_name == "train":
        print()

### Data Engineering


In [ ]:
train_df["second"] = train_df["timestep_time"].astype(int)
train_df["hour"] = (train_df["timestep_time"] // 3600).astype(int)
train_secondly = (
    train_df.groupby("second")
    .agg(avg_speed_kmh=("speed_kmh", "mean"), vehicle_count=("vehicle_id", "nunique"))
    .reset_index()
)
train_hourly = (
    train_df.groupby("hour")
    .agg(avg_speed_kmh=("speed_kmh", "mean"), vehicle_count=("vehicle_id", "nunique"))
    .reset_index()
)

test_df["second"] = test_df["timestep_time"].astype(int)
test_df["hour"] = (test_df["timestep_time"] // 3600).astype(int)
test_secondly = (
    test_df.groupby("second")
    .agg(avg_speed_kmh=("speed_kmh", "mean"), vehicle_count=("vehicle_id", "nunique"))
    .reset_index()
)
test_hourly = (
    test_df.groupby("hour")
    .agg(avg_speed_kmh=("speed_kmh", "mean"), vehicle_count=("vehicle_id", "nunique"))
    .reset_index()
)

### Speed Distribution


In [ ]:
for df, df_name in [(train_df, "train"), (test_df, "test")]:
    plt.figure(figsize=(6, 4))
    plt.hist(df["speed_kmh"], bins=30)
    plt.title(f"Speed Distribution ({df_name})")
    plt.xlabel("Speed (km/h)")
    plt.ylabel("Frequency")
    plt.show()

### Average Speed per Second


In [ ]:
for df, df_name in [(train_secondly, "train"), (test_secondly, "test")]:
    plt.figure(figsize=(10, 4))
    plt.plot(df["second"], df["avg_speed_kmh"])
    plt.title(f"Average Speed per Second ({df_name})")
    plt.xlabel("Second")
    plt.ylabel("Speed (km/h)")
    plt.show()

### Vehicle Count per Second


In [ ]:
for df, df_name in [(train_secondly, "train"), (test_secondly, "test")]:
    plt.figure(figsize=(10, 4))
    plt.plot(df["second"], df["vehicle_count"])
    plt.title(f"Vehicle Count per Second ({df_name})")
    plt.xlabel("Second")
    plt.ylabel("Count")
    plt.show()

### Average Speed per Hour


In [ ]:
for df, df_name in [(train_hourly, "train"), (test_hourly, "test")]:
    plt.figure(figsize=(6, 4))
    plt.plot(df["hour"], df["avg_speed_kmh"], marker="o")
    plt.title(f"Average Speed per Hour ({df_name})")
    plt.xlabel("Hour")
    plt.ylabel("Speed (km/h)")
    plt.show()

### Vehicle Count per Hour


In [ ]:
for df, df_name in [(train_hourly, "train"), (test_hourly, "test")]:
    plt.figure(figsize=(6, 4))
    plt.plot(df["hour"], df["vehicle_count"], marker="o")
    plt.title(f"Vehicle Count per Hour ({df_name})")
    plt.xlabel("Hour")
    plt.ylabel("Count")
    plt.show()

### Vehicle Count and Speed over Time


In [ ]:
for df, df_name in [(train_secondly, "train"), (test_secondly, "test")]:
    fig, ax1 = plt.subplots(figsize=(10, 4))
    ax1.plot(df["second"], df["vehicle_count"], label="Vehicle Count")
    ax1.set_xlabel("Second")
    ax1.set_ylabel("Count")
    ax2 = ax1.twinx()
    ax2.plot(df["second"], df["avg_speed_kmh"], label="Avg Speed (km/h)", color="orange")
    ax2.set_ylabel("Speed (km/h)")
    fig.legend(loc="upper right")
    plt.title(f"Vehicle Count and Speed over Time ({df_name})")
    plt.show()

### Average Speed and Traffic Generation Periods per Hour


In [ ]:
for df, df_name, periods in [
    (train_hourly, "train", TRAIN_TRAFFIC_GENERATION_PERIODS),
    (test_hourly, "test", TEST_TRAFFIC_GENERATION_PERIODS),
]:
    fig, ax1 = plt.subplots(figsize=(10, 4))
    ax1.plot(df["hour"], df["avg_speed_kmh"], marker="o", label="Avg Speed (km/h)")
    ax1.set_xlabel("Hour")
    ax1.set_ylabel("Avg Speed (km/h)")
    ax2 = ax1.twinx()
    ax2.plot(df["hour"], periods, marker="o", label="Gen. Period (s)", color="orange")
    ax2.set_ylabel("Traffic Gen Period (s)")
    fig.legend(loc="upper right")
    plt.title(f"Average Speed and Traffic Generation Periods per Hour ({df_name})")
    plt.show()

## Conversion to CSV


To convert the xml output files to csv, we can use the [xml2csv.py](https://sumo.dlr.de/docs/Tools/Xml.html#xml2csvpy) tool of SUMO.


In [ ]:
# !python $SUMO_HOME/tools/xml/xml2csv.py athens-10h/dump-train-athens-10h.xml
# !python $SUMO_HOME/tools/xml/xml2csv.py athens-10h/emission-train-athens-10h.xml
# !python $SUMO_HOME/tools/xml/xml2csv.py athens-10h/fcd-train-athens-10h.xml

In [ ]:
# !python $SUMO_HOME/tools/xml/xml2csv.py athens-10h/dump-test-athens-10h.xml
# !python $SUMO_HOME/tools/xml/xml2csv.py athens-10h/emission-test-athens-10h.xml
# !python $SUMO_HOME/tools/xml/xml2csv.py athens-10h/fcd-test-athens-10h.xml

In [ ]:
# !rm athens-10h/dump-train-athens-10h.xml
# !rm athens-10h/emission-train-athens-10h.xml
# !rm athens-10h/fcd-train-athens-10h.xml
# !rm athens-10h/dump-test-athens-10h.xml
# !rm athens-10h/emission-test-athens-10h.xml
# !rm athens-10h/fcd-test-athens-10h.xml

## Concept Drift


### Ideas

1. Disable single edge (road closure)
1. Disable multiple edges (cycling event)
1. Disable single lane (accident)
1. Disable multiple lanes (road works)
1. Increase of traffic on certain area (concert or football match)
1. Increase of traffic on whole area (holidays)
1. Reduction of traffic on certain area (shut down of a factory)
1. Reduction of traffic on whole area (holidays)
1. Global reduction of speed (weather)
1. Global increase of vehicle following gaps (weather)
1. Global decrease of acceleration (weather)
1. Global increase of deceleration (weather)

For 1-4 we can use an additional file named `closure.add.xml` that contains the edge/lane closures. This file can be then added to the additional files of the simulation as follows:

```xml
<additional-files value="osm.poly.xml.gz, vtypes.xml, closure.add.xml"/>
```

For 9-12 we can use a new vehicle type called `car-rain` that has the following changes:

- `accel` = 2.0 (default is 2.6)
- `decel` = 4.0 (default is 4.5)
- `minGap` = 3.5 (default is 2.5)
- `maxSpeed` = 33.33 (default is 55.55)
- `speedFactor` = 0.8 (default is 1.0)


### Closure Drift


In [ ]:
print("Generating closure train random trips...")
generate_random_trips("closure-train", TRAIN_TRAFFIC_GENERATION_PERIODS, 42)
print("Generating closure test random trips...")
generate_random_trips("closure-test", TEST_TRAFFIC_GENERATION_PERIODS, 123)

In [ ]:
print("Updating closure train trip IDs...")
update_trip_ids("closure-train")
print("Updating closure test trip IDs...")
update_trip_ids("closure-test")

In [ ]:
print("Updating closure train vehicle types...")
update_vehicle_types("closure-train")
print("Updating closure test vehicle types...")
update_vehicle_types("closure-test")

In [ ]:
!sumo -c athens-10h/closure-train.sumocfg

In [ ]:
!sumo -c athens-10h/closure-test.sumocfg

In [ ]:
print("Parsing closure train FCD output...")
closure_train_df = parse_fcd_output_xml(CLOSURE_TRAIN_FCD_DATASET_PATH)
print("Parsing closure test FCD output...")
closure_test_df = parse_fcd_output_xml(CLOSURE_TEST_FCD_DATASET_PATH)

In [ ]:
closure_train_df = closure_train_df.dropna().reset_index(drop=True)
closure_test_df = closure_test_df.dropna().reset_index(drop=True)

closure_train_df = closure_train_df[closure_train_df["timestep_time"] < 36000].reset_index(drop=True)
closure_test_df = closure_test_df[closure_test_df["timestep_time"] < 36000].reset_index(drop=True)

for df, df_name in [(closure_train_df, "closure-train"), (closure_test_df, "closure-test")]:
    print(f"Dataset: {df_name}")
    print(f"Data shape: {df.shape}")
    print(f"Average speed: {df['speed_kmh'].mean():.2f} km/h")
    print(f"Average distance (max odometer): {df.groupby('vehicle_id')['odometer'].max().mean():.2f} m")
    print(f"Unique vehicles: {df['vehicle_id'].nunique()}")
    print(f"Time span: {df['timestep_time'].min():.2f} s to {df['timestep_time'].max():.2f} s")
    if df_name == "closure-train":
        print()

In [ ]:
closure_train_df["second"] = closure_train_df["timestep_time"].astype(int)
closure_train_df["hour"] = (closure_train_df["timestep_time"] // 3600).astype(int)
closure_train_secondly = (
    closure_train_df.groupby("second")
    .agg(avg_speed_kmh=("speed_kmh", "mean"), vehicle_count=("vehicle_id", "nunique"))
    .reset_index()
)
closure_train_hourly = (
    closure_train_df.groupby("hour")
    .agg(avg_speed_kmh=("speed_kmh", "mean"), vehicle_count=("vehicle_id", "nunique"))
    .reset_index()
)

closure_test_df["second"] = closure_test_df["timestep_time"].astype(int)
closure_test_df["hour"] = (closure_test_df["timestep_time"] // 3600).astype(int)
closure_test_secondly = (
    closure_test_df.groupby("second")
    .agg(avg_speed_kmh=("speed_kmh", "mean"), vehicle_count=("vehicle_id", "nunique"))
    .reset_index()
)
closure_test_hourly = (
    closure_test_df.groupby("hour")
    .agg(avg_speed_kmh=("speed_kmh", "mean"), vehicle_count=("vehicle_id", "nunique"))
    .reset_index()
)

In [ ]:
for df, df_name in [(closure_train_df, "closure-train"), (closure_test_df, "closure-test")]:
    plt.figure(figsize=(6, 4))
    plt.hist(df["speed_kmh"], bins=30)
    plt.title(f"Speed Distribution ({df_name})")
    plt.xlabel("Speed (km/h)")
    plt.ylabel("Frequency")
    plt.show()

In [ ]:
for df, df_name in [(closure_train_secondly, "closure-train"), (closure_test_secondly, "closure-test")]:
    plt.figure(figsize=(10, 4))
    plt.plot(df["second"], df["avg_speed_kmh"])
    plt.title(f"Average Speed per Second ({df_name})")
    plt.xlabel("Second")
    plt.ylabel("Speed (km/h)")
    plt.show()

In [ ]:
for df, df_name in [(closure_train_secondly, "closure-train"), (closure_test_secondly, "closure-test")]:
    plt.figure(figsize=(10, 4))
    plt.plot(df["second"], df["vehicle_count"])
    plt.title(f"Vehicle Count per Second ({df_name})")
    plt.xlabel("Second")
    plt.ylabel("Count")
    plt.show()

In [ ]:
for df, df_name in [(closure_train_hourly, "closure-train"), (closure_test_hourly, "closure-test")]:
    plt.figure(figsize=(6, 4))
    plt.plot(df["hour"], df["avg_speed_kmh"], marker="o")
    plt.title(f"Average Speed per Hour ({df_name})")
    plt.xlabel("Hour")
    plt.ylabel("Speed (km/h)")
    plt.show()

In [ ]:
for df, df_name in [(closure_train_hourly, "closure-train"), (closure_test_hourly, "closure-test")]:
    plt.figure(figsize=(6, 4))
    plt.plot(df["hour"], df["vehicle_count"], marker="o")
    plt.title(f"Vehicle Count per Hour ({df_name})")
    plt.xlabel("Hour")
    plt.ylabel("Count")
    plt.show()

In [ ]:
for df, df_name in [(closure_train_secondly, "closure-train"), (closure_test_secondly, "closure-test")]:
    fig, ax1 = plt.subplots(figsize=(10, 4))
    ax1.plot(df["second"], df["vehicle_count"], label="Vehicle Count")
    ax1.set_xlabel("Second")
    ax1.set_ylabel("Count")
    ax2 = ax1.twinx()
    ax2.plot(df["second"], df["avg_speed_kmh"], label="Avg Speed (km/h)", color="orange")
    ax2.set_ylabel("Speed (km/h)")
    fig.legend(loc="upper right")
    plt.title(f"Vehicle Count and Speed over Time ({df_name})")
    plt.show()

In [ ]:
for df, df_name, periods in [
    (closure_train_hourly, "closure-train", TRAIN_TRAFFIC_GENERATION_PERIODS),
    (closure_test_hourly, "closure-test", TEST_TRAFFIC_GENERATION_PERIODS),
]:
    fig, ax1 = plt.subplots(figsize=(10, 4))
    ax1.plot(df["hour"], df["avg_speed_kmh"], marker="o", label="Avg Speed (km/h)")
    ax1.set_xlabel("Hour")
    ax1.set_ylabel("Avg Speed (km/h)")
    ax2 = ax1.twinx()
    ax2.plot(df["hour"], periods, marker="o", label="Gen. Period (s)", color="orange")
    ax2.set_ylabel("Traffic Gen Period (s)")
    fig.legend(loc="upper right")
    plt.title(f"Average Speed and Traffic Generation Periods per Hour ({df_name})")
    plt.show()

### Rain Drift


In [ ]:
print("Generating rain train random trips...")
generate_random_trips("rain-train", TRAIN_TRAFFIC_GENERATION_PERIODS, 42)
print("Generating rain test random trips...")
generate_random_trips("rain-test", TEST_TRAFFIC_GENERATION_PERIODS, 123)

In [ ]:
print("Updating rain train trip IDs...")
update_trip_ids("rain-train")
print("Updating rain test trip IDs...")
update_trip_ids("rain-test")

In [ ]:
print("Updating rain train vehicle types...")
update_vehicle_types("rain-train")
print("Updating rain test vehicle types...")
update_vehicle_types("rain-test")

In [ ]:
print("Generating rain fixed routes...")
generate_fixed_routes("rain-fixed")

In [ ]:
!sumo -c athens-10h/rain-train.sumocfg

In [ ]:
!sumo -c athens-10h/rain-test.sumocfg

In [ ]:
print("Parsing rain train FCD output...")
rain_train_df = parse_fcd_output_xml(RAIN_TRAIN_FCD_DATASET_PATH)
print("Parsing rain test FCD output...")
rain_test_df = parse_fcd_output_xml(RAIN_TEST_FCD_DATASET_PATH)

In [ ]:
rain_train_df = rain_train_df.dropna().reset_index(drop=True)
rain_test_df = rain_test_df.dropna().reset_index(drop=True)

rain_train_df = rain_train_df[rain_train_df["timestep_time"] < 36000].reset_index(drop=True)
rain_test_df = rain_test_df[rain_test_df["timestep_time"] < 36000].reset_index(drop=True)

for df, df_name in [(rain_train_df, "rain-train"), (rain_test_df, "rain-test")]:
    print(f"Dataset: {df_name}")
    print(f"Data shape: {df.shape}")
    print(f"Average speed: {df['speed_kmh'].mean():.2f} km/h")
    print(f"Average distance (max odometer): {df.groupby('vehicle_id')['odometer'].max().mean():.2f} m")
    print(f"Unique vehicles: {df['vehicle_id'].nunique()}")
    print(f"Time span: {df['timestep_time'].min():.2f} s to {df['timestep_time'].max():.2f} s")
    if df_name == "rain-train":
        print()

In [ ]:
rain_train_df["second"] = rain_train_df["timestep_time"].astype(int)
rain_train_df["hour"] = (rain_train_df["timestep_time"] // 3600).astype(int)
rain_train_secondly = (
    rain_train_df.groupby("second")
    .agg(avg_speed_kmh=("speed_kmh", "mean"), vehicle_count=("vehicle_id", "nunique"))
    .reset_index()
)
rain_train_hourly = (
    rain_train_df.groupby("hour")
    .agg(avg_speed_kmh=("speed_kmh", "mean"), vehicle_count=("vehicle_id", "nunique"))
    .reset_index()
)

rain_test_df["second"] = rain_test_df["timestep_time"].astype(int)
rain_test_df["hour"] = (rain_test_df["timestep_time"] // 3600).astype(int)
rain_test_secondly = (
    rain_test_df.groupby("second")
    .agg(avg_speed_kmh=("speed_kmh", "mean"), vehicle_count=("vehicle_id", "nunique"))
    .reset_index()
)
rain_test_hourly = (
    rain_test_df.groupby("hour")
    .agg(avg_speed_kmh=("speed_kmh", "mean"), vehicle_count=("vehicle_id", "nunique"))
    .reset_index()
)

In [ ]:
for df, df_name in [(rain_train_df, "rain-train"), (rain_test_df, "rain-test")]:
    plt.figure(figsize=(6, 4))
    plt.hist(df["speed_kmh"], bins=30)
    plt.title(f"Speed Distribution ({df_name})")
    plt.xlabel("Speed (km/h)")
    plt.ylabel("Frequency")
    plt.show()

In [ ]:
for df, df_name in [(rain_train_secondly, "rain-train"), (rain_test_secondly, "rain-test")]:
    plt.figure(figsize=(10, 4))
    plt.plot(df["second"], df["avg_speed_kmh"])
    plt.title(f"Average Speed per Second ({df_name})")
    plt.xlabel("Second")
    plt.ylabel("Speed (km/h)")
    plt.show()

In [ ]:
for df, df_name in [(rain_train_secondly, "rain-train"), (rain_test_secondly, "rain-test")]:
    plt.figure(figsize=(10, 4))
    plt.plot(df["second"], df["vehicle_count"])
    plt.title(f"Vehicle Count per Second ({df_name})")
    plt.xlabel("Second")
    plt.ylabel("Count")
    plt.show()

In [ ]:
for df, df_name in [(rain_train_hourly, "rain-train"), (rain_test_hourly, "rain-test")]:
    plt.figure(figsize=(6, 4))
    plt.plot(df["hour"], df["avg_speed_kmh"], marker="o")
    plt.title(f"Average Speed per Hour ({df_name})")
    plt.xlabel("Hour")
    plt.ylabel("Speed (km/h)")
    plt.show()

In [ ]:
for df, df_name in [(rain_train_hourly, "rain-train"), (rain_test_hourly, "rain-test")]:
    plt.figure(figsize=(6, 4))
    plt.plot(df["hour"], df["vehicle_count"], marker="o")
    plt.title(f"Vehicle Count per Hour ({df_name})")
    plt.xlabel("Hour")
    plt.ylabel("Count")
    plt.show()

In [ ]:
for df, df_name in [(rain_train_secondly, "rain-train"), (rain_test_secondly, "rain-test")]:
    fig, ax1 = plt.subplots(figsize=(10, 4))
    ax1.plot(df["second"], df["vehicle_count"], label="Vehicle Count")
    ax1.set_xlabel("Second")
    ax1.set_ylabel("Count")
    ax2 = ax1.twinx()
    ax2.plot(df["second"], df["avg_speed_kmh"], label="Avg Speed (km/h)", color="orange")
    ax2.set_ylabel("Speed (km/h)")
    fig.legend(loc="upper right")
    plt.title(f"Vehicle Count and Speed over Time ({df_name})")
    plt.show()

In [ ]:
for df, df_name, periods in [
    (rain_train_hourly, "rain-train", TRAIN_TRAFFIC_GENERATION_PERIODS),
    (rain_test_hourly, "rain-test", TEST_TRAFFIC_GENERATION_PERIODS),
]:
    fig, ax1 = plt.subplots(figsize=(10, 4))
    ax1.plot(df["hour"], df["avg_speed_kmh"], marker="o", label="Avg Speed (km/h)")
    ax1.set_xlabel("Hour")
    ax1.set_ylabel("Avg Speed (km/h)")
    ax2 = ax1.twinx()
    ax2.plot(df["hour"], periods, marker="o", label="Gen. Period (s)", color="orange")
    ax2.set_ylabel("Traffic Gen Period (s)")
    fig.legend(loc="upper right")
    plt.title(f"Average Speed and Traffic Generation Periods per Hour ({df_name})")
    plt.show()